# chap6. 비모수검정

비모수검정은 모수에 대한 가설없이 가설검정하는 것, 분포무관(Distribution-free) 검정

- 모수검정: 모집단의 분포에 대한 가정 하 모수에 대해 가설설정 후 표본의 정보를 사용하여 검정
- 비모수검정: 가정없이 표본의 정보를 사용하여 "분포의 형태가 동일한지" 여부를 검정
    
- 자료가 정규분포가 아닐 때, 표본의 크기가 작은 경우, 자료가 서열척도 또는 명목척도인 경우 사용

| **표본 종류**   | **표본 개수** | **비모수 검정 (서열척도)** | **비모수 검정 (명목척도)**           | **모수 검정 (명목 척도)**                   |
| ----------- | --------- | ----------------- | --------------------------- | ----------------------------------- |
| 1표본         | 1         | 부호검정, Wilcoxon 부호순위 검정  | 적합성 검정, Run 검정       | 일표본 t검정              |
| 2표본 독립      | 2         | 순위합 검정, Mann–Whitney U 검정 | 카이제곱 독립성 검정, 동질성 검정  | 독립표본 t검정   |
| 3개 이상 독립 집단 | ≥ 3       | Kruskal–Wallis 검정 | 카이제곱 독립성 검정, 동질성 검정                 | 일원분산분석 (ANOVA)                      |
| 2표본 대응      | 2         | 부호검정, Wilcoxon 부호순위 검정  | McNemar 검정                  | 대응표본 t검정                          |
| 3개 이상 대응 집단 | ≥ 3       | Friedman 검정       | Cochran Q 검정 | 반복측정 분산분석 (Repeated Measures ANOVA) |

## 6-1. 카이제곱 검정: 카이제곱 분포

**적합성 검정: 다항모집단 비율의 차이**  
적합도 검정은 관측값들이 어떤 이론이나 이론적 분포를 따르고 있는지 검정하는 것.  
카이제곱분포를 이용한 검정은 기대도수가 적어도 5이상이 될 때 적용해야 함

<가설설정>  
- H0: 구해진 도수분포의 도수와 이론도수의 차이는 없다.
- H1: 구해진 도수분포의 도수와 이론도수의 차이가 있다.

| 항목       | 설명                                                      |
| -------- | ------------------------------------------------------- |
| ✅ 목적     | 한 개의 **범주형 변수**의 분포가 특정한 \*\*이론적 분포(기대비율)\*\*와 일치하는지 검정 |
| ✅ 검정 방법  | `scipy.stats.chisquare()` 사용                            |
| ✅ 예시     | 동전을 100번 던졌더니 앞면이 48번 나옴 → **공정한지** 검정                  |
| ✅ 기대값    | 이론적으로 주어짐 (예: 1:1, 1:2:1 등)                             |
| ✅ 데이터 구조 | **단일 범주형 변수**의 빈도표                                      |


In [1]:
# 적합성 검정: 세 후보자의 지지도가 다르다고 할 수 있나? 없다.
import numpy as np
from scipy.stats import chi2, chisquare

dt = np.array([60, 50, 40])
print(dt)

m0 = dt.mean()#평균지지도(비교군)
c_stat, p = chisquare(dt, m0)
print(f"검정통계량: {c_stat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

[60 50 40]
검정통계량: 4.0 / p-value: 0.135
유의수준 5% 하 p-value 0.1353352832366127 이므로 귀무가설을 기각할 수 없다.


**독립성 검정: 한 모집단 내 여러 수준의 차이**  
독립성 검정(혹은 교차분석)은 다수의 인자들에 의해 분할되어 있는 데이터에서 인자들이 관찰값에 영향을 주고 있는지 여부 검정  
<조건>
- 자유도가 1인 경우 전체 데이터수가 30보다 크면서 각 칸의 빈도가 5 이상일 때 사용
- 데이터수가 30보다 크면서 5 미만의 기대빈도의 칸이 전체 칸의 20%보다 적고, 모든 칸에 1 이상의 기대빈도가 있다면 척도에 관계없이 사용
- 각 칸의 기대빈도가 5미만인 경우, 변수들의 범주를 묶거나 이항검정법을 사용
- 도수가 작아도 **피셔의 정확검정**을 이용하면 집계표의 독립성 검정 가능

<가설검정>
- H0: 두 인자는 독립이다.(연관이 없다.)
- H1: 두 인자는 독립이 아니다.(연관이 있다.)

| 항목       | 설명                                     |
| -------- | -------------------------------------- |
| ✅ 목적     | 두 개의 **범주형 변수** 간에 **독립인지 연관이 있는지** 검정 |
| ✅ 검정 방법  | `scipy.stats.chi2_contingency()` 사용    |
| ✅ 예시     | 성별과 흡연여부가 **관련이 있는지** 검정               |
| ✅ 기대값    | 관측된 행/열의 합으로부터 계산                      |
| ✅ 데이터 구조 | **2차원 교차표(Contingency Table)**         |


In [2]:
# 성별과 안경 착용여부가 서로 독립인지 여부를 유의수준 5% 하 검정: 독립이 아니다.(연관성 있음)
import pandas as pd
from scipy.stats import chi2_contingency

dt = pd.DataFrame({"성별":["남자", "여자"], "안경o":[10, 30], "안경x":[40, 20]}).set_index("성별") #교차표
c_stat, p, dof, expected = chi2_contingency(dt, correction=False) #2x2 표일 경우, **연속성 보정(Yates' correction)**을 적용할지 여부
print(f"검정통계량: {c_stat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

검정통계량: 16.666666666666668 / p-value: 0.000
유의수준 5% 하 p-value 4.455709060405612e-05 이므로 귀무가설을 기각할 수 있다.


In [3]:
# Fisher's exact test
tb = pd.DataFrame({"op":["A", "B"], "승":[10, 3], "패":[2, 5]}).set_index("op") #교차표
display(tb)

from scipy.stats import fisher_exact

f_stat, p = fisher_exact(tb, alternative='greater') #right(A의 실력이 더 좋다.)
print(f"검정통계량: {f_stat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

,승,패
op,,
A,10,2
B,3,5


검정통계량: 8.333333333333334 / p-value: 0.052
유의수준 5% 하 p-value 0.05211558307533548 이므로 귀무가설을 기각할 수 없다.


**동질성 검정: 여러 (부)모집단 간 여러 수준에 대한 차이**  
속성 A, B를 가진 부모집단들로부터 정해진 표본의 크기만큼 자료를 추출하는 경우, 분할표에서 부모집단의 비율이 동일한지 여부 검정

<가설설정>
- H0: 모든 집단의 분포가 차이가 없다.(동일하다.)
- H1: 적어도 한 집단은 분포 상 서로 차이가 없다.(동일하지 않다.)

| 항목           | 설명                                                  |
| ------------ | --------------------------------------------------- |
| ✅ **목적**     | 두 개 이상의 **집단이 어떤 범주형 특성 분포에서 같은 분포를 가지는지(동일한지)** 검정 |
| ✅ **사용 함수**  | `scipy.stats.chi2_contingency()`                    |
| ✅ **데이터 구조** | 교차표 (범주형 변수 2개: 집단 × 특성)                            |
| ✅ **예시**     | 서울·부산·광주 학생들의 스마트폰 기종 비율이 **같은지 검정**                |


In [4]:
# 프로그램에 대한 연령층별 시청자들의 선호가 다른지 유의수준 5%로 검정: 변수는 tv와 연령(연령에 따라 tv 프로그램 선호도 다름)
tb = pd.DataFrame({"TV":['A', 'B', 'C'], "청년": [120, 30, 50], "중년": [10, 75, 15], "장년": [10, 30, 60]}).set_index('TV')
display(tb)
c_stat, p, dof, expected = chi2_contingency(tb, correction=False)
print(f"검정통계량: {c_stat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

,청년,중년,장년
TV,,,
A,120,10,10
B,30,75,30
C,50,15,60


검정통계량: 180.49523809523808 / p-value: 0.000
유의수준 5% 하 p-value 5.836850688101578e-38 이므로 귀무가설을 기각할 수 있다.


## 6-2. Run 검정: Run 검정표, Z분포

**일표본 Run 검정**  
한 개의 샘플이 무작위로 추출되었는지 여부를 검정한다. Run이란 동일한 관측값이 연속적으로 이어진 것을 말한다.  
(한 종류의 부호 혹은 한 집단이 시작하여 끝날 때까지 한 덩어리를 의미)  

- 범주형 데이터의 경우 각 범주의 개수와 run의 개수를 사용하여 검정 진행
- 수치형 데이터의 경우 중앙값을 기준으로 데이터를 이진화 한 후 검정 진행

<가설설정>
- H0: 샘플이 무작위와 차이가 없다.(무작위로 추출)
- H1: 샘플이 무작위와 차이가 있다.(무작위로 추출되지 않음)

In [5]:
# 범주형 데이터: 샘플이 무작위로 추출되었다. 
from collections import Counter
dt = ['a']*2+['b']*2+['a']*4+['b']*4+['a', 'b']+['a']*2+['b']*2+['a']*2+['b']*3+['a']*2+['b']*2+['a', 'b']
Counter(dt)
dt = np.where([x == 'a' for x in dt], 1, 0)

from statsmodels.sandbox.stats.runs import runstest_1samp
zstat, p = runstest_1samp(dt)
print(f"검정통계량: {zstat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

검정통계량: -0.3721438547031917 / p-value: 0.710
유의수준 5% 하 p-value 0.7097857460625617 이므로 귀무가설을 기각할 수 없다.


In [6]:
# 수치형 데이터: 샘플이 무작위로 추출되었다.
dt = [50, 60,70, 40, 30, 20, 10, 70, 80, 100]

from statsmodels.sandbox.stats.runs import runstest_1samp

zstat, p = runstest_1samp(dt, cutoff='median')
print(f"검정통계량: {zstat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

검정통계량: -1.0062305898749053 / p-value: 0.314
유의수준 5% 하 p-value 0.31430466047385397 이므로 귀무가설을 기각할 수 없다.


**이표본 Run 검정**  
<가설검정>
- H0: 두 데이터는 같은 분포에서 왔다. 
- H1: 두 데이터는 다른 분포에서 왔다.

In [7]:
#수치형 데이터: 두 데이터는 다른 분포에서 왔다.
dt1 = [23, 42, 36, 27, 48, 52, 35, 31]
dt2 = [43, 56, 38, 20, 46, 51, 36]

dt1 = list(map(lambda x: float(x), dt1))
dt2 = list(map(lambda x: float(x), dt2))

from statsmodels.sandbox.stats.runs import runstest_2samp

zstat, p = runstest_2samp(dt1, dt2)
print(f"검정통계량: {zstat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

ties detected
검정통계량: 0.01794895396544339 / p-value: 0.986
유의수준 5% 하 p-value 0.9856795756756671 이므로 귀무가설을 기각할 수 없다.


## 6-3. 이항변수 데이터 검정: 카이제곱 분포
**맥니머 검정**  
이항변수인 두 변수의 대응관계가 있는 데이터 분포의 차이를 검정할 때 사용

<가설검정>
- H0: 두 변수의 데이터 분포는 차이가 없다.
- H1: 두 변수의 데이터 분포는 차이가 있다.

In [8]:
#프로모션 행사 전후로 상품에 대한 흥미 유무: 분포가 동일
tb = pd.DataFrame([[9, 12], [24, 35]], index=['전_있음', '전_없음'], columns=['후_있음', '후_없음'])
display(tb)

from statsmodels.stats.contingency_tables import mcnemar

mc = mcnemar(tb.values, exact=False, correction=False) #2*2, 정확 검정 (binomial test) 여부, 연속성 보정 적용 여부(샘플 작을때만)
print(f"검정통계량: {mc.statistic} / p-value: {mc.pvalue:.3f}")
print(f"유의수준 5% 하 p-value {mc.pvalue} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

,후_있음,후_없음
전_있음,9,12
전_없음,24,35


검정통계량: 4.0 / p-value: 0.046
유의수준 5% 하 p-value 0.04550026389635857 이므로 귀무가설을 기각할 수 없다.


**코크란 Q검정**  
이항변수인 세 변수 이상의 대응관계가 있는 데이터 분포의 차이 검정

<가설검정>
- H0: 각 변수의 데이터 분포는 차이가 없다. 
- H1: 적어도 한 쌍의 변수의 데이터 분포는 차이가 있다.

| 항목            | **코크란의 Q 검정**                    | **일원배치 분산분석 (One-way ANOVA)** |
| ------------- | -------------------------------- | ----------------------------- |
| **검정 종류**     | **비모수 검정**                       | **모수 검정**                     |
| **데이터 유형**    | **이진형** 종속변수 (예/아니오), 대응(반복측정)자료 | **연속형** 종속변수, 독립된 집단          |
| **독립변수 수준 수** | 3개 이상 처리 조건                      | 3개 이상 그룹/처리                   |
| **대응 여부**     | **대응 표본** (같은 대상이 여러 조건 실험)      | **독립 표본** (집단 간 비교)           |
| **가정**        | 정규성, 등분산 필요 없음                   | 정규성, 등분산성, 독립성 필요             |
| **통계량 분포**    | 카이제곱 근사                          | F-분포                          |
| **사용 목적**     | 세 가지 이상 조건에서 성공률 차이 비교           | 세 그룹 이상 평균 차이 검정              |
| **사후검정 가능성**  | 제한적 (McNemar test 등으로 쌍 비교)      | Tukey, Bonferroni 등 사후검정 가능   |


In [9]:
# 연예인 3명에 대한 호감도 데이터를 얻기 위해 8명에게 설문조사를 실시했을 때, 연예인에 대한 호감도 비율에 차이가 있는가?
# 호감도 비율의 차이가 있다.(적어도 한 쌍은 다르다.)
temp = [[0, 1, 0, 1, 0, 0, 0, 0], 
       [1, 1, 0, 1, 0, 0, 1, 1], 
       [0, 1, 1, 1, 1, 1, 1, 1]]
dt = pd.DataFrame(temp, index=['가수1', '가수2', '가수3'], columns=np.arange(1, 9, 1)).T
display(dt)

from statsmodels.stats.contingency_tables import cochrans_q

ccq = cochrans_q(dt)
stat, p = ccq.statistic, ccq.pvalue

print(f"검정통계량: {stat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

,가수1,가수2,가수3
1,0,1,0
2,1,1,1
3,0,0,1
4,1,1,1
5,0,0,1
6,0,0,1
7,0,1,1
8,0,1,1


검정통계량: 6.333333333333333 / p-value: 0.042
유의수준 5% 하 p-value 0.042143843509276406 이므로 귀무가설을 기각할 수 있다.


In [10]:
# 모든 쌍의 맥니머 검정으로 어느 쌍 사이의 분포차이가 있는지 확인

pr = [['가수1', '가수2'], ['가수1', '가수3'], ['가수2', '가수3']]

for pair in pr:
    dt1 = dt[pair]
    tb = pd.crosstab(dt1[pair[0]], dt1[pair[1]])
    mc = mcnemar(tb, exact=False, correction=False)
    stat, p = mc.statistic, mc.pvalue
    print(f"{pair[0]} - {pair[1]} 검정통계량: {stat} / p-value: {p:.3f}", '***' if p <= 0.05 else '')

가수1 - 가수2 검정통계량: 3.0 / p-value: 0.083 
가수1 - 가수3 검정통계량: 5.0 / p-value: 0.025 ***
가수2 - 가수3 검정통계량: 1.0 / p-value: 0.317 


## 6-4. 부호, 순위 데이터 검정
**일표본 부호 검정: 이항분포, Z분포**  
n <= 100 이면, 부호검정 통계량은 이항분포를 따르고,  
n > 100 이면, 정규화한 부호검정은 정규분포를 따른다. 일표본 t검정에 대응한다.

<가설검정(예)>
- H0: 데이터의 중앙값은 200과 차이가 없다. 
- H1: 데이터의 중앙값은 200과 차이가 있다.

In [11]:
# 데이터의 중앙값으로 알려진 m0 = 200일 때, 가설검정

dt = np.array([203, 204, 197, 195, 201, 205, 198, 199, 194, 207])
m0 = 200

#데이터의 크기가 100 이하이므로 이항분포 따름
# 검정통계량 B는 B(n, p=0.5) 인 이항분포(n은 plus 수)

from statsmodels.stats.descriptivestats import sign_test

stat, p = sign_test(dt, mu0=m0) # alternative='two-sided', 'larger', 'smaller'(실제 중앙값 기준)

print(f"검정통계량: {stat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

검정통계량: 0.0 / p-value: 1.000
유의수준 5% 하 p-value 1.0 이므로 귀무가설을 기각할 수 없다.


**이표본 부호 검정: 이항분포, Z분포**  
n<= 100이면, 부호검정 통계량은 이항분포를 따르고, n>100이면 정규화한 부호검정 통계량은 정규분포를 따른다.  
대응표본 t 검정에 대응한다.

In [12]:
# A, B의 5점 척도 만족도 설문조사로부터 A의 만족도가 더 높다고 할 수 있는지 검정
from scipy.stats import binom

# 데이터
dt1 = np.array([4, 3, 5, 2, 1, 3, 4, 3])
dt2 = np.array([3, 2, 3, 1, 2, 2, 2, 2])
diff = dt1 - dt2
m0 = 0

# 부호검정 구성
plus = np.sum(diff > m0)
minus = np.sum(diff < m0)
n = plus + minus  # 동점은 제외
print(f"plus: {plus}, minus: {minus}, n: {n}") 

# 귀무가설 하 B(n, p=0.5), 단측검정: A > B → 오른쪽 꼬리검정
p_value = binom.sf(plus-1, n, 0.5)  # P(X ≥ plus)

print(f"유의수준 5% 하 p-value {p_value} 이므로 귀무가설을 기각할 수", "있다." if p_value <= 0.05 else "없다.")

plus: 7, minus: 1, n: 8
유의수준 5% 하 p-value 0.03515625 이므로 귀무가설을 기각할 수 있다.


**일표본 윌콕스 부호순위 검정: 윌콕슨 부호순위 검정표, Z 분포**  
n <= 20 이면, 윌콕슨 순위합 분포를 따르고, n > 20 이면 정규분포에 근사한다.  
일표본 t검정에 대응한다. 

In [13]:
# 데이터의 중앙값으로 알려진 m0=200 일때 가설검정
#H0: 데이터의 중앙값은 200과 차이가 없다. / H1: 데이터의 중앙값은 200과 차이가 있다.
dt = np.array([203, 204, 197, 195, 201, 205, 198, 199, 194, 207])
m0 = 200

from scipy.stats import wilcoxon

stat, p = wilcoxon([m0]*len(dt), dt)

print(f"검정통계량: {stat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

검정통계량: 25.5 / p-value: 0.846
유의수준 5% 하 p-value 0.845703125 이므로 귀무가설을 기각할 수 없다.


**이표본 윌콕슨 부호순위 검정: 윌콕슨 부호순위 검정표, z분포**  
n <= 20 이면, 윌콕슨 순위합 분포를 따르고, n > 20 이면 정규분포에 근사한다.  
대응표본 t검정에 대응한다. 

In [14]:
# 동일한 피험자 8명에게 맥박을 2번 측정하였을 때 1, 2번째 측정값 차이여부
dt1 = np.array([79, 96, 85, 69, 88, 75, 83, 88])
dt2 = np.array([70, 88, 73, 74, 75, 79, 77, 81])

stat, p = wilcoxon(dt1, dt2, zero_method='wilcox')
print(f"검정통계량: {stat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

검정통계량: 3.0 / p-value: 0.039
유의수준 5% 하 p-value 0.0390625 이므로 귀무가설을 기각할 수 있다.


**윌콕슨 순위합 검정(맨 휘트니 u검정): 윌콕슨 순위합 검정표, Z분포**  
n(=n1+n2) <= 25 이면 순위합 분포를 따르고, n > 25 이면 정규분포에 근사한다.  
독립표본 t 검정에 대응한다. 맨 위트니 검정과 동일한 결과를 얻는다.

In [15]:
# 팀별 영업성적이 차이가 있는지 검정(중앙값 차이는 0이다, 아니다.)
dt1 = [87, 75, 65, 95, 90, 81, 93]
dt2 = [57, 85, 90, 83, 87, 71]

from scipy.stats import ranksums, mannwhitneyu

zstat, p = ranksums(dt1, dt2)
print("<순위합 검정>")
print(f"검정통계량: {zstat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

mstat, p2 = mannwhitneyu(dt1, dt2)
print("<맨 휘트니 검정>")
print(f"검정통계량: {mstat} / p-value: {p2:.3f}")
print(f"유의수준 5% 하 p-value {p2} 이므로 귀무가설을 기각할 수", "있다." if p2 <= 0.05 else "없다.")

<순위합 검정>
검정통계량: 0.8571428571428571 / p-value: 0.391
유의수준 5% 하 p-value 0.39136593830755195 이므로 귀무가설을 기각할 수 없다.
<맨 휘트니 검정>
검정통계량: 27.0 / p-value: 0.431
유의수준 5% 하 p-value 0.43076586096421876 이므로 귀무가설을 기각할 수 없다.


## k 표본 순위 데이터 검정
**크러스컬 월리스 검정: 크러스컬 월리스 검정표, 카이제곱 분포**  
세 변수 이상의 대응관계가 없는 데이터의 차이를 검정한다.(대응관계 없는 일원배치 분산분석 비모수 버전)  
사후검정은 **윌콕슨 순위합 검정**으로 진행 가능 

3개 범주이고 데이터 크기가 15 이하이거나, 4개 범주이고 데이터 크기가 14개 이하인 작은 표본은 크러스컬 월리스 검정표를 사용하고  
이외 데이터 크기가 충분히 크면 검정통계량이 카이제곱분포에 근사한다.

In [18]:
# A, B, C 세 사람의 모의고사 성적을 통해 성취도의 차이가 있는지 검정(분포의 차이?)
# H0: 세 사람의 성취도는 차이가 없다. / H1: 적어도 한 쌍의 성취도는 차이가 있다. : 적어도 한 쌍의 성취도는 차이가 있음
dt = pd.DataFrame([[69, 67, 65, 59, 66], 
                   [56, 63, 55, 40], 
                   [71, 72, 70, 75]], index=['A', 'B', 'C'])
display(dt)

from scipy.stats import kruskal
stat, p = kruskal(dt.values[0], dt.values[1], dt.values[2], nan_policy='omit') # nan 값을 제거하고 자동계산

print(f"검정통계량: {stat} / p-value: {p:.3f}")
print(f"유의수준 5% 하 p-value {p} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

,0,1,2,3,4
A,69,67,65,59,66.0
B,56,63,55,40,NaN
C,71,72,70,75,NaN


검정통계량: 10.117582417582419 / p-value: 0.006
유의수준 5% 하 p-value 0.006353234607321947 이므로 귀무가설을 기각할 수 있다.


In [23]:
# 사후검정
col_comp = [['A', 'B'], ['B', 'C'], ['A', 'C']]

from scipy.stats import ranksums

for s1, s2 in col_comp:
    stat, p = ranksums(dt.loc[s1], dt.loc[s2])
    print(f"<{s1} - {s2} 순위합 검정(사후)>")
    print(f"검정통계량: {stat} / p-value: {p:.3f}", "***" if p <= 0.05 else '')
    print(f"유의수준 5% 하 p-value {p:.3f} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.\n")

<A - B 순위합 검정(사후)>
검정통계량: 1.3578057164544433 / p-value: 0.175 
유의수준 5% 하 p-value 0.175 이므로 귀무가설을 기각할 수 없다.

<B - C 순위합 검정(사후)>
검정통계량: -1.775592090748118 / p-value: 0.076 
유의수준 5% 하 p-value 0.076 이므로 귀무가설을 기각할 수 없다.

<A - C 순위합 검정(사후)>
검정통계량: -2.6111648393354674 / p-value: 0.009 ***
유의수준 5% 하 p-value 0.009 이므로 귀무가설을 기각할 수 있다.


**프리드먼 검정: 프리드먼 검정표, 카이제곱 분포**  
세 변수 이상의 대응관계가 있는 데이터의 차이를 검정한다.(대응관계 있는 일원배치 분산분석의 비모수 버전)  
사후검정은 윌콕슨 부호순위검정으로 진행  

3개 범주이며 데이터 크기가 9 이하이거나, 4개 범주이며 데이터 크기가 5 이하인 작은 표본은 프리드먼 검정표 사용,  
데이터 크기가 충분히 크면 검정통계량 Q가 카이제곱 분포에 근사

| 항목           | 크루스칼-왈리스 검정             | 프리드먼 검정                             |
| ------------ | ----------------------- | ----------------------------------- |
| 설계           | 독립된 여러 집단 비교            | 같은 집단의 반복 측정 비교                     |
| 독립성          | 집단 간 독립                 | 집단 간 비독립 (같은 대상 반복 측정)              |
| 유사한 대응 모수 검정 | 일원분산분석 (ANOVA)          | 반복측정 분산분석 (Repeated-measures ANOVA) |
| 사용 예         | A, B, C 세 사람의 독립된 시험 점수 | 한 사람이 3가지 방식으로 측정된 점수               |
| 데이터 형태       | 집단별 나열                  | 개인별로 행, 조건별로 열                      |


In [26]:
# 운전자 A, B, C, D 의 운전 점수에 차이가 있는지 검정: 차이 없다.
# H0: 네 운전자의 운전 점수는 차이가 없다.  / H1: 적어도 한 쌍의 운전자의 운전 점수는 차이가 있다. 
from scipy.stats import friedmanchisquare

dt = pd.DataFrame([[4, 2, 5], [3, 5, 2], [5, 4, 4], [1, 1, 3]], index=['A', 'B', 'C', 'D'])
display(dt) # 3번 반복측정으로 봄

stat, p = friedmanchisquare(dt.values[0], dt.values[1], dt.values[2], dt.values[3])
print(f"검정통계량: {stat} / p-value: {p:.3f}", "***" if p <= 0.05 else '')
print(f"유의수준 5% 하 p-value {p:.3f} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

,0,1,2
A,4,2,5
B,3,5,2
C,5,4,4
D,1,1,3


검정통계량: 4.200000000000003 / p-value: 0.241 
유의수준 5% 하 p-value 0.241 이므로 귀무가설을 기각할 수 없다.


In [28]:
# 사후검정: 윌콕슨 부호검정
from itertools import combinations

col_comp = list(combinations(dt.index, 2)) #2개씩 쌍

from scipy.stats import wilcoxon

for s1, s2 in col_comp:
    stat, p = wilcoxon(dt.loc[s1], dt.loc[s2], zero_method='wilcox')
    print(f"{s1} - {s2} 검정통계량: {stat} / p-value: {p:.3f}", "***" if p <= 0.05 else '')    

A - B 검정통계량: 2.5 / p-value: 1.000 
A - C 검정통계량: 1.5 / p-value: 0.500 
A - D 검정통계량: 0.0 / p-value: 0.250 
B - C 검정통계량: 1.0 / p-value: 0.500 
B - D 검정통계량: 1.0 / p-value: 0.500 
C - D 검정통계량: 0.0 / p-value: 0.250 


## 연습문제
**1.아래 그래프는 A, B, C 동별 입주민의 주민대표 찬반투표 결과를 나타낸다. 동별 찬반 비율이 동일한지 귀무가설과 대립가설을  
설정하고, 검정통계량을 계산하여 검정하시오.(유의수준 0.05)**

| 구분            | 독립성 검정                | 동질성 검정              |
| ------------- | --------------------- | ------------------- |
| **목적**        | 두 범주형 변수 간에 독립적인지 검정  | 두 집단 이상의 비율이 같은지 검정 |
| **가설**        | 변수 A와 변수 B는 서로 독립이다   | 여러 집단의 비율 분포가 동일하다  |
| **데이터 수집 방식** | 단일 집합에서 두 변수를 관측      | 서로 다른 집단에서 같은 변수 관측 |
| **예시**        | "성별과 흡연 여부가 관련이 있는가?" | "세 지역의 찬반 비율이 같은가?" |


In [32]:
# 동질성 검정 문제이다.[범주-범주 비율분포 비교]
#H0: 각 동별 찬반비율은 모두 같다. / H1: 각 동별 찬반비율은 한 쌍이라도 다르다.

from scipy.stats import chi2_contingency

dt = pd.DataFrame([[50, 60, 65], [45, 32, 55]], index=['찬성', '반대'], columns=['A', 'B', 'C'])
display(dt)

stat, p, dof, expected = chi2_contingency(dt)

print(f"검정통계량: {stat:.3f} / p-value: {p:.3f}", "***" if p <= 0.05 else '')
print(f"유의수준 5% 하 p-value {p:.3f} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")
print("따라서, 유의수준 5% 하, 각 동별 찬반 비율은 모두 같다.")

,A,B,C
찬성,50,60,65
반대,45,32,55


검정통계량: 3.667 / p-value: 0.160 
유의수준 5% 하 p-value 0.160 이므로 귀무가설을 기각할 수 없다.
따라서, 유의수준 5% 하, 각 동별 찬반 비율은 모두 같다.


**2.특정 캠페인에 노출되기 전과 후의 캠페인 주제에 대한 지지여부를 기록한 데이터이다. 캠페인 전후의 지지비율이   
차이가 있는지 귀무가설과 대립가설을 설정하고, 검정통계량을 계산하여 검정하시오.(유의수준 0.05)**

In [42]:
# 대응표본이며, 전 후 비율 차이 검정이므로 맥니머 검정이다.
# H0: 캠페인 전후의 지지비율 차이가 없다. / H1: 캠페인 전후 지지비율 차이가 있다.
dt = pd.read_csv('https://raw.githubusercontent.com/algoboni/pythoncodebook1-1/main/practice6_ba.csv', index_col=0)
ct = pd.crosstab(dt['before'], dt['after'])

from statsmodels.stats.contingency_tables import mcnemar

mc = mcnemar(ct, exact=False, correction=False)

stat, p = mc.statistic, mc.pvalue

print(f"검정통계량: {stat:.3f} / p-value: {p:.3f}", "***" if p <= 0.05 else '')
print(f"유의수준 5% 하 p-value {p:.3f} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")
print("따라서, 유의수준 5% 하, 캠페인 전 후의 지지비율 차이가 없다.")

검정통계량: 0.429 / p-value: 0.513 
유의수준 5% 하 p-value 0.513 이므로 귀무가설을 기각할 수 없다.
따라서, 유의수준 5% 하, 캠페인 전 후의 지지비율 차이가 없다.


**3.다음의 T와 F가 무작위로 나열되어 있다고 볼 수 있는지 귀무가설과 대립가설을 설정하고, 검정통계량을 계산하여  
검정하시오.(유의수준 0.05)**

In [58]:
# 일표본 run 검정
# H0: 표본은 무작위로 나열되어 있다. / H1: 표본은 무작위로 나열되지 않았다.

from statsmodels.sandbox.stats.runs import runstest_1samp

dt = ['T', 'F', 'F', 'T', 'F', 'T', 'F', 'T', 'T', 'F', 'F', 'T', 'F', 'T', 'F', 'T', 'F', 'T']

dt2 = np.where([x == 'T' for x in dt], 1, 0)

stat, p = runstest_1samp(dt2)

print(f"검정통계량: {stat:.3f} / p-value: {p:.3f}", "***" if p <= 0.05 else '')
print(f"유의수준 5% 하 p-value {p:.3f} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")
print("따라서, 유의수준 5% 하, 표본은 무작위로 나열되지 않았다.")

검정통계량: 2.187 / p-value: 0.029 ***
유의수준 5% 하 p-value 0.029 이므로 귀무가설을 기각할 수 있다.
따라서, 유의수준 5% 하, 표본은 무작위로 나열되지 않았다.


**4. 20대보다 30대가 기대하는 연봉상승률이 더 높다는 주장이 있다. 이 주장이 타당한지 확인해보기 위해서 20대 10명과 30대 8명을  
임의로 추출하여 기대하는 연봉상승률을 다음과 같이 정리하였다. 가설을 설정하고 검정통계량을 계산하여 검정하시오.(유의수준 5%)**

왜? 윌콕슨 순위합 검정?? 맨 휘트니 u검정(독립표본 t분포에 대응)

| 조건       | 설명                            |
| -------- | ----------------------------- |
| **두 집단** | 서로 독립된 20대와 30대               |
| **표본 수** | 작음 (n₁=10, n₂=8)              |
| **정규성**  | 보장되지 않음 → 비모수 검정 필요           |
| **목표**   | 두 집단의 위치 차이 (중앙값 또는 순위 중심) 검정 |


In [88]:
# H0: 20대와 30대의 연봉 상승률은 같다. / H1: 20대보다 30대의 연봉 상승률이 더 높다.: 30대가 더 높다.
a = [3.0, 3.5, 2.0, 2.8, 5.0, 0, 2.3, 2.8, 3.3, 3.5]
b = [3.5, 5,5, 5.0, 5.0, 10.0, 8.0, 2.5, 3.0]

display(dt)
from scipy.stats import mannwhitneyu, ranksums
stat, p = mannwhitneyu(a, b, alternative='less') # 20 < 30(음수)
stat2, p2 = ranksums(a, b, alternative='less')

print(f"검정통계량: {stat:.3f} / p-value: {p:.3f}", "***" if p <= 0.05 else '')
print(f"검정통계량: {stat2:.3f} / p-value: {p2:.3f}", "***" if p2 <= 0.05 else '')
print(f"유의수준 5% 하 p-value {p:.3f} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")

,20대,30대
0,3.0,3.5
1,3.5,5.0
2,2.0,5.0
3,2.8,5.0
4,5.0,5.0
5,0.0,10.0
6,2.3,8.0
7,2.8,2.5
8,3.3,3.0
9,3.5,NaN


검정통계량: 15.500 / p-value: 0.008 ***
검정통계량: -2.409 / p-value: 0.008 ***
유의수준 5% 하 p-value 0.008 이므로 귀무가설을 기각할 수 있다.


**5. 양식, 한식, 중식에 대해 5명의 선호도를 나타내는 표이다. '선호하지 않는다' 를 1, '보통이다'를 2, '선호한다'를 3으로 응답  
하였다. 음식 종류에 따라 선호도 차이가 있는지를 귀무가설과 대립가설로 설정하고 검정통계량을 계산하여 검정하시오.(유의수준 5%)**

In [94]:
# 프리드먼 검정: 대응집단 간 순위(선호도) 차이 비교 
#H0: 음식 종류에 따라 선호도 차이가 없다. / H1: 음식 종류에 따라 선호도 차이가 있다.
dt = pd.DataFrame([[1, 3, 1, 3, 1], 
                  [1, 3, 3, 1, 1], 
                  [2, 3, 3, 3, 1]], index=['양식', '한식', '중식'], columns=['A', 'B', 'C', 'D', 'E'])

display(dt)

from scipy.stats import friedmanchisquare

stat, p = friedmanchisquare(dt.loc['양식'].values, dt.loc['한식'].values, dt.loc['중식'].values)
print(f"검정통계량: {stat:.3f} / p-value: {p:.3f}", "***" if p <= 0.05 else '')
print(f"유의수준 5% 하 p-value {p:.3f} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")
print("따라서, 음식 종류에 따라 선호도 차이가 없다.")

,A,B,C,D,E
양식,1,3,1,3,1
한식,1,3,3,1,1
중식,2,3,3,3,1


검정통계량: 2.667 / p-value: 0.264 
유의수준 5% 하 p-value 0.264 이므로 귀무가설을 기각할 수 없다.
따라서, 음식 종류에 따라 선호도 차이가 없다.


**6. 임의로 추출한 A, B, C 지역 학생들의 키가 다음과 같다. 지역별 학생들의 키가 차이가 있는지 검정하기 위한 귀무가설과 대립가설을  
설정하고, 검정통계량을 계산하여 검정하시오.(유의수준 5%)**

In [97]:
#임의로 추출하였기 때문에 독립성이 있다. 세 변수 이상의 대응표본이 아닌 데이터 간 차이를 검정하는 문제로 크러스컬 월리스 검정
# H0: 지역별 학생들의 키가 차이가 없다. / H1: 지역별 학생들의 키가 차이가 있다.

from scipy.stats import kruskal

a = [177, 167, 188, 189, 152, 159, 184, 175]
b = [151, 177, 150, 187, 167, 166, 179, 161, 174]
c = [173, 151, 156, 182, 188, 175, 150, 165, 176, 183]

stat, p = kruskal(a, b, c, nan_policy='omit')

print(f"검정통계량: {stat:.3f} / p-value: {p:.3f}", "***" if p <= 0.05 else '')
print(f"유의수준 5% 하 p-value {p:.3f} 이므로 귀무가설을 기각할 수", "있다." if p <= 0.05 else "없다.")
print("따라서, 지역별 학생들의 키가 차이가 없다.")

검정통계량: 1.331 / p-value: 0.514 
유의수준 5% 하 p-value 0.514 이므로 귀무가설을 기각할 수 없다.
따라서, 지역별 학생들의 키가 차이가 없다.
